# 덜읽더알 과거 뉴스 RAG — 유사도 구간 검증

**배경**: STEP2 `fetch_related_news`는 과거 뉴스 검색 결과 중 `30 <= int((1 - score) * 100) <= 75`인 것만 골라 상위 3건을 인사이트 생성에 씁니다.
"유사도가 너무 높으면 단순 중복 기사만 나온다"는 문제를 해결하려고 정한 구간이에요.

**이 노트북이 확인하는 것**
1. **지표 정의 검증**: `score`가 정말 코사인 유사도인가? (ChromaDB 기본 거리 = L2)
2. **구간별 품질**: 실제 코사인 유사도 구간마다 과거 기사가 *단순 중복 / 맥락 연결 / 무관* 중 무엇인지
3. **정책 비교**: 필터 없이 상위 3건 vs 30~75% 필터 → 중복률·맥락 연결률·결과 없음 비율
4. **판정 신뢰도**: GPT 판정 20쌍을 다른 회사의 LLM(Claude)이 독립적으로 판정해 교차 검증
5. **근본 원인 분석**: 같은 아티스트 기준으로 커버리지와 검색 변별력을 분리하고, 임베딩 설정 3종 비교 (LLM 호출 없음)

## 준비
- 🔑 보안 비밀에 `OPENROUTER_API_KEY` 등록 + 노트북 액세스 켜기 (판정용 LLM 호출 약 400회, 몇백 원 수준)
- 위에서부터 차례로 실행

In [ ]:
!pip -q install chromadb==1.5.9 sentence-transformers langchain-openai pandas scikit-learn
!git clone -q --depth 1 https://github.com/yuwls00/RLKM rlkm || true
!rm -rf chroma_copy && cp -r rlkm/chroma_db chroma_copy   # 원본을 건드리지 않도록 복사본 사용

## 1. 데이터 불러오기 (실제 서비스와 같은 방식으로 질의 생성)

In [ ]:
import json, sqlite3, random
import numpy as np, pandas as pd, chromadb
from sentence_transformers import SentenceTransformer

client = chromadb.PersistentClient(path="chroma_copy")
past = client.get_collection("past_news")
print("past_news 벡터 수:", past.count())

conn = sqlite3.connect("rlkm/k_enter_news.db"); conn.row_factory = sqlite3.Row
def _parse(x):
    try: return json.loads(x) if x else []
    except Exception: return []
def _summary(x):
    items = _parse(x)
    return " ".join(i.get("content","") for i in items if isinstance(i, dict))[:300]

cur_rows = conn.execute("SELECT id, ko_title, keywords, summary FROM processed_news WHERE importance IS NOT NULL AND ko_title IS NOT NULL").fetchall()
past_info = {r["id"]: {"title": r["ko_title"] or "", "summary": _summary(r["summary"])}
             for r in conn.execute("SELECT id, ko_title, summary FROM past_news")}

N_QUERIES = 40
random.seed(42)
sample = random.sample(cur_rows, N_QUERIES)
print("현재 뉴스", len(cur_rows), "건 중", N_QUERIES, "건 샘플")

# STEP2/vectorstore.py 와 같은 임베딩 모델
model = SentenceTransformer("Snowflake/snowflake-arctic-embed-m")

past_news 벡터 수: 162
현재 뉴스 341 건 중 40 건 샘플


In [ ]:
rows = []
for r in sample:
    kw = _parse(r["keywords"])
    query_text = (r["ko_title"] or "") + " " + " ".join(kw)       # rag_search.py 201행과 동일
    q = model.encode(query_text)
    res = past.query(query_embeddings=[q.tolist()], n_results=10, include=["distances","metadatas","embeddings"])
    for rank, (d, meta, e) in enumerate(zip(res["distances"][0], res["metadatas"][0], res["embeddings"][0]), 1):
        e = np.array(e)
        rows.append(dict(
            query_id=r["id"], query_title=r["ko_title"], query_summary=_summary(r["summary"]),
            rank=rank, past_id=meta.get("id"), past_title=past_info.get(meta.get("id"),{}).get("title", meta.get("title","")),
            past_summary=past_info.get(meta.get("id"),{}).get("summary",""),
            score=d,
            code_pct=int((1 - d) * 100),                          # 코드가 쓰던 '유사도 %'
            cosine=float(q @ e / np.linalg.norm(q) / np.linalg.norm(e)),  # 진짜 코사인 유사도
            q_norm=float(np.linalg.norm(q)), d_norm=float(np.linalg.norm(e)),
        ))
pairs = pd.DataFrame(rows)
print(len(pairs), "쌍 (질의 40 × 상위 10)")

400 쌍 (질의 40 × 상위 10)


## 2. 지표 정의 검증 — `score`는 코사인 유사도인가?

In [ ]:
print("컬렉션 거리 방식:", past.configuration_json.get("hnsw", {}).get("space"))
print("임베딩 길이(정규화 여부): 질의", pairs.q_norm.round(3).unique()[:3], "/ 문서", pairs.d_norm.round(3).unique()[:3])
sq = 2 - 2 * pairs.cosine
print("score == 2 - 2·cos (제곱 L2) 인가?", np.allclose(pairs.score, sq, atol=1e-3))
print("code_pct ≈ (2·cos - 1)×100 인가?", np.allclose(pairs.code_pct, (2*pairs.cosine - 1)*100, atol=1.01))
print()
print("→ 코드의 30% = 코사인", (1+0.30)/2, " / 75% = 코사인", (1+0.75)/2)
print("상위 10건의 코사인 범위:", round(pairs.cosine.min(), 3), "~", round(pairs.cosine.max(), 3))

컬렉션 거리 방식: l2
임베딩 길이(정규화 여부): 질의 [1.] / 문서 [1.]
score == 2 - 2·cos (제곱 L2) 인가? True
code_pct ≈ (2·cos - 1)×100 인가? True

→ 코드의 30% = 코사인 0.65  / 75% = 코사인 0.875
상위 10건의 코사인 범위: 0.7 ~ 0.981


## 3. LLM으로 쌍별 관계 판정
- **duplicate**: 같은 사건·발표를 다룬 기사 (새로운 맥락 정보 없음)
- **context**: 같은 아티스트·작품·이슈의 *다른* 사건 → 현재 뉴스의 배경·흐름을 설명해 줌
- **unrelated**: 관련 없음

In [ ]:
from google.colab import userdata
from langchain_openai import ChatOpenAI
from concurrent.futures import ThreadPoolExecutor

judge = ChatOpenAI(model="openai/gpt-5.4-mini", openai_api_key=userdata.get("OPENROUTER_API_KEY"),
                   openai_api_base="https://openrouter.ai/api/v1", temperature=0, request_timeout=30, max_tokens=1024)

JUDGE_PROMPT = """두 K-엔터 뉴스의 관계를 판정하세요.

[현재 뉴스] {qt}
요약: {qs}

[과거 뉴스] {pt}
요약: {ps}

라벨 정의:
- duplicate: 같은 사건·발표·기록을 다룬 기사. 현재 뉴스를 이해하는 데 새 정보를 주지 않음
- context: 같은 아티스트·작품·이슈에 대한 '다른' 사건. 현재 뉴스의 배경이나 흐름을 설명해 줌
- unrelated: 의미 있는 관련이 없음

JSON으로만 답하세요: {{"label": "duplicate|context|unrelated", "reason": "한 문장"}}"""

def judge_pair(row):
    try:
        out = judge.invoke(JUDGE_PROMPT.format(qt=row.query_title, qs=row.query_summary, pt=row.past_title, ps=row.past_summary)).content.strip()
        if out.startswith("```"): out = out.split("```")[1].removeprefix("json")
        j = json.loads(out)
        return j.get("label","error"), j.get("reason","")
    except Exception as e:
        return "error", type(e).__name__

with ThreadPoolExecutor(max_workers=8) as ex:
    results = list(ex.map(judge_pair, pairs.itertuples()))
pairs["label"] = [r[0] for r in results]; pairs["reason"] = [r[1] for r in results]
pairs.to_csv("rag_band_pairs.csv", index=False, encoding="utf-8-sig")
print(pairs.label.value_counts())
print("판정 실패 비율:", (pairs.label=="error").mean())   # 0이 아니면 크레딧/키 확인

label
unrelated    383
context       14
duplicate      3
Name: count, dtype: int64
판정 실패 비율: 0.0


## 4. 결과 — 코사인 구간별 관계 분포

In [ ]:
bins = [0, 0.65, 0.75, 0.80, 0.85, 0.875, 0.90, 1.0]
pairs["cos_bin"] = pd.cut(pairs.cosine, bins=bins)
pairs["in_code_band"] = pairs.code_pct.between(30, 75)
ok = pairs[pairs.label != "error"]
dist = pd.crosstab(ok.cos_bin, ok.label, normalize="index").round(3)
dist["건수"] = ok.groupby("cos_bin").size()
dist

label,context,duplicate,unrelated,건수
cos_bin,,,,
"(0.65, 0.75]",0.000,0.00,1.000,14
"(0.75, 0.8]",0.333,0.00,0.667,9
"(0.8, 0.85]",0.136,0.00,0.864,44
"(0.85, 0.875]",0.000,0.00,1.000,8
"(0.875, 0.9]",0.079,0.00,0.921,38
"(0.9, 1.0]",0.007,0.01,0.983,287


## 5. 정책 비교 — 실제로 사용자에게 보여지는 상위 3건 기준

In [ ]:
def policy(df, mask=None, k=3):
    sel = df if mask is None else df[mask]
    top = sel.sort_values(["query_id","rank"]).groupby("query_id").head(k)
    n_q = df.query_id.nunique()
    return pd.Series({
        "선택된 기사 수": len(top),
        "중복률": (top.label=="duplicate").mean(),
        "맥락 연결률": (top.label=="context").mean(),
        "무관 비율": (top.label=="unrelated").mean(),
        "결과 0건 질의 비율": 1 - top.query_id.nunique()/n_q,
    })
cmp = pd.DataFrame({
    "A. 필터 없음 (상위 3)":      policy(ok),
    "B. 코드 30~75% (cos .65~.875)": policy(ok, ok.code_pct.between(30, 75)),
    "C. cos 0.75~0.875":          policy(ok, ok.cosine.between(0.75, 0.875)),
    "D. cos ≥ 0.875만":            policy(ok, ok.cosine >= 0.875),
})
cmp.to_csv("rag_band_policy.csv", encoding="utf-8-sig")
cmp.style.format("{:.3f}")

,A. 필터 없음 (상위 3),B. 코드 30~75% (cos .65~.875),C. cos 0.75~0.875,D. cos ≥ 0.875만
선택된 기사 수,120.000,30.000,23.000,100.000
중복률,0.025,0.000,0.000,0.030
맥락 연결률,0.042,0.100,0.130,0.020
무관 비율,0.933,0.900,0.870,0.950
결과 0건 질의 비율,0.000,0.750,0.775,0.150


## 6. 판정 신뢰도 — 이종 LLM 교차 검증
구간별로 고르게 뽑은 20쌍을 **다른 회사의 LLM(Claude)이 GPT 판정을 보지 않은 상태로** 독립 판정했습니다.
`CROSS_LABELS`가 Claude의 판정, `label`이 GPT의 판정입니다.

In [ ]:
pool = ok.groupby("cos_bin", group_keys=False, observed=True).apply(lambda g: g.sample(min(len(g), 5), random_state=0))
check = pool.sample(n=min(20, len(pool)), random_state=0).reset_index(drop=True)   # 구간별로 고르게 20쌍
for i, r in check.iterrows():
    print(f"[{i}] 현재: {r.query_title}\n     과거: {r.past_title}\n     과거 요약: {r.past_summary[:120]}\n")

[0] 현재: KickFlip, ‘Music Bank’에서 두 번째 트로피 수상…TXT, SANTOS BRAVOS 등 화려한 무대 선보여
     과거: 동남아시아 K-pop 콘서트 및 투어
     과거 요약: Southeast Asia에서 K-pop 콘서트 및 투어에 대한 정보를 제공합니다. Bandwagon은 2011년부터 동남아시아에서 K-pop 콘서트 및 투어를 다루는 가장 많이 인용되는 출처 중 하나입니다. 싱가포

[1] 현재: 한채아, 44세 안 믿기는 수영복 몸매 ‘부러워’
     과거: 김성철, 진기주와 함께 새 드라마 ‘슬리핑 닥터’ 출연 확정
     과거 요약: 김성철, 진기주 주연 확정, 새 드라마 ‘슬리핑 닥터’ 출연. KBS2 드라마 ‘슬리핑 닥터’는 홍경(진기주)이 주인공인 로맨틱 코미디 드라마. 김성철은 이번에 처음으로 의사 역할을 맡게 된다.

[2] 현재: 유연석 주연 ‘phantom lawyer’, 이덕화 방어에 힘쓰다
     과거: 박지훈, ‘키친 솔져의 전설’ 티저 공개: 요리 실력으로 팬심 사로잡아
     과거 요약: 새 드라마 ‘키친 솔져의 전설’의 새로운 티저가 공개되었습니다. ‘키친 솔져의 전설’은 인기 웹 소설 및 웹툰을 기반으로 한 군사 판타지 코미디 드라마로, 군대 요리사 강성재(박지훈)의 이야기를 담고 있습니다. 티저

[3] 현재: ‘출생 아동’ 스캔들 재연…K팝 인기男 솔로 가수, 자녀 숨기고 금전 지원 | 스타뉴스
     과거: 충남 배경 드라마 ‘트웰브’, 23일 KBS·디즈니+ 동시 공개
     과거 요약: '트웰브'가 23일 KBS·디즈니+에서 동시 공개된다. 촬영은 충남 논산시 등 도내 여러 지역에서 진행되었다. 마동석, 박형식, 서인국, 성동일, 이주빈 등이 출연한다. 동양의 12지신을 모티브로 한 판타지 액션물로

[4] 현재: 임영웅, 상상도 못한 행보…놀라운 인연 with 남도형
     과거: 걸스데이 티파니, 배우 윤요한 결혼…‘깊은 신뢰와 사랑’ 바탕
     과거 요약:

In [ ]:
# Claude 독립 판정 (위 20쌍 순서, d=duplicate / c=context / u=unrelated)
CROSS_LABELS = ["u","u","u","u","u",  "u","u","c","u","u",
         "c","u","u","u","u",  "u","u","u","u","u"]

In [ ]:
from sklearn.metrics import cohen_kappa_score
m = {"d":"duplicate","c":"context","u":"unrelated"}
h = [m.get(x.strip().lower(), None) for x in CROSS_LABELS][:len(check)]
assert all(h), "CROSS_LABELS를 모두 d/c/u로 채워주세요"
check["claude"] = h
print("LLM–LLM 일치율:", round((check.claude == check.label).mean(), 3))
print("Cohen's kappa:", round(cohen_kappa_score(check.claude, check.label), 3))
check[check.claude != check.label][["query_title","past_title","cosine","label","claude","reason"]]

LLM–LLM 일치율: 0.95
Cohen's kappa: 0.773


,query_title,past_title,cosine,label,claude,reason
9,"April 멤버, 브랜드 평판 순위 발표… Wanna One 박지훈 이어 상위권","박지훈, ‘키친 솔져의 전설’ 티저 공개: 요리 실력으로 팬심 사로잡아",0.898533,context,unrelated,"두 기사 모두 박지훈을 공통으로 다루지만, 하나는 브랜드 평판 순위 발표이고 다른 ..."


## 7. 근본 원인 분석 — 커버리지와 변별력 분리 + 임베딩 모델 비교
실시간 뉴스 특성상 과거 데이터에 관련 기사가 애초에 없는 경우가 많아, "검색이 못 찾은 것"과 "찾을 게 없던 것"을 분리합니다.
- **근사 정답**: 현재 뉴스와 과거 뉴스의 `artist_tags`가 하나라도 겹치면 관련 후보
- **Hit@3**: 관련 기사가 있는 질의에서 상위 3건 안에 찾은 비율
- **쌍 단위 AUC**: 점수가 같은 아티스트 기사를 위로 올리는 정도 (0.5 = 무작위)
- **Top1 AUC**: 1등 점수만으로 관련 기사 유무를 구분할 수 있는지 (임계값 필터 가능성)

In [ ]:
# ===== 7. 아티스트 기준 실험: 커버리지와 변별력 분리 + 임베딩 모델 비교 =====
from sklearn.metrics import roc_auc_score

def _tags(x):
    return {t.strip().lower() for t in _parse(x) if isinstance(t, str) and t.strip()}

past_rows = conn.execute("SELECT id, ko_title, keywords, artist_tags FROM past_news").fetchall()
past_ids  = [r["id"] for r in past_rows]
past_tags = [_tags(r["artist_tags"]) for r in past_rows]
# 저장 당시 문서 내용 그대로 재현 (vectorstore.py 와 동일한 형식)
past_docs = [f"{r['ko_title'] or ''}\n\n아티스트: {', '.join(_parse(r['artist_tags']))}\n키워드: {', '.join(_parse(r['keywords']))}" for r in past_rows]

all_cur = conn.execute("SELECT id, ko_title, keywords, artist_tags FROM processed_news WHERE importance IS NOT NULL AND ko_title IS NOT NULL").fetchall()
queries = [(r["ko_title"] or "") + " " + " ".join(_parse(r["keywords"])) for r in all_cur]
q_tags  = [_tags(r["artist_tags"]) for r in all_cur]
rel = np.array([[len(qt & pt) > 0 for pt in past_tags] for qt in q_tags])   # 같은 아티스트 = 관련 후보
has_rel = rel.any(axis=1)
print(f"현재 뉴스 {len(all_cur)}건 중 같은 아티스트의 과거 기사가 있는 비율(커버리지): {has_rel.mean():.1%}")

def evaluate(name, q_emb, d_emb):
    q = q_emb / np.linalg.norm(q_emb, axis=1, keepdims=True)
    d = d_emb / np.linalg.norm(d_emb, axis=1, keepdims=True)
    S = q @ d.T                                   # 코사인 유사도 행렬 (질의 × 과거기사)
    order = np.argsort(-S, axis=1)
    hit3 = np.array([rel[i, order[i, :3]].any() for i in range(len(S))])
    top1 = S[np.arange(len(S)), order[:, 0]]
    return pd.Series({
        "Hit@3 (관련 있는 질의)": hit3[has_rel].mean(),
        "쌍 단위 AUC (점수→같은 아티스트)": roc_auc_score(rel.ravel(), S.ravel()),
        "Top1 점수 AUC (관련 있음/없음 구분)": roc_auc_score(has_rel, top1),
        "Top1 코사인 평균: 관련 있음": top1[has_rel].mean(),
        "Top1 코사인 평균: 관련 없음": top1[~has_rel].mean(),
        "전체 코사인 평균": S.mean(),
    }, name=name)

results = []
# (1) 실제 서비스 설정: arctic-embed-m, 접두어 없음
results.append(evaluate("arctic (서비스 그대로)", model.encode(queries), model.encode(past_docs)))
# (2) arctic + 모델 권장 질의 접두어
PFX = "Represent this sentence for searching relevant passages: "
results.append(evaluate("arctic + 질의 접두어", model.encode([PFX + t for t in queries]), model.encode(past_docs)))
# (3) 다국어 모델 (한국어 지원)
m_e5 = SentenceTransformer("intfloat/multilingual-e5-base")
results.append(evaluate("multilingual-e5-base", m_e5.encode(["query: " + t for t in queries]), m_e5.encode(["passage: " + t for t in past_docs])))

model_cmp = pd.DataFrame(results).T
model_cmp.to_csv("rag_model_compare.csv", encoding="utf-8-sig")
model_cmp.style.format("{:.3f}")

현재 뉴스 341건 중 같은 아티스트의 과거 기사가 있는 비율(커버리지): 22.6%


,arctic (서비스 그대로),arctic + 질의 접두어,multilingual-e5-base
Hit@3 (관련 있는 질의),0.338,0.442,0.688
쌍 단위 AUC (점수→같은 아티스트),0.538,0.780,0.841
Top1 점수 AUC (관련 있음/없음 구분),0.492,0.642,0.832
Top1 코사인 평균: 관련 있음,0.918,0.367,0.837
Top1 코사인 평균: 관련 없음,0.926,0.333,0.811
전체 코사인 평균,0.746,0.247,0.768


## 정리
- **지표 정의**: 코드의 "유사도 30~75%"는 제곱 L2 거리 기반 값이며, 정규화 임베딩에서 코사인 0.65~0.875에 해당
- **구간 필터**: 맥락 연결률 4.2% → 10.0%, 대신 질의의 75%에서 결과 0건
- **근본 원인**: 관련 과거 기사 커버리지 22.6% + 영어 중심 임베딩의 낮은 변별력 (AUC 0.54)
- **개선 검증**: 질의 접두어 적용 시 AUC 0.78, multilingual-e5-base 교체 시 Hit@3 33.8% → 68.8%, AUC 0.84
- 결과 CSV: `results/` 폴더 / 전체 해석: [README](README.md)